## Example 1: Compatibility Layer Functionality

In [ ]:
import inspect
import sys

import notebook_shim

# Example 1: Compatibility Layer Functionality
print("Exploring notebook_shim compatibility layer")

try:
    # Show notebook_shim information
    print(f"notebook_shim version: {getattr(notebook_shim, '__version__', 'unknown')}")
    print(f"Module path: {notebook_shim.__file__}")
    print(f"Python version: {sys.version.split()[0]}")

    # The shim's three submodules: the mixin that routes classic NotebookApp options,
    # the NotebookApp traits it keeps, and the helpers for classic server extensions
    from notebook_shim import nbserver, shim, traits

    print("\nShim components:")
    assert inspect.isclass(shim.NotebookConfigShimMixin), shim.NotebookConfigShimMixin
    mixin_methods = {'shim_config_from_notebook_to_jupyter_server', 'update_config'}
    assert mixin_methods <= set(vars(shim.NotebookConfigShimMixin)), sorted(vars(shim.NotebookConfigShimMixin))
    print(f"  shim.NotebookConfigShimMixin: class defining {sorted(mixin_methods)}")

    assert inspect.isclass(traits.NotebookAppTraits), traits.NotebookAppTraits
    print(f"  traits.NotebookAppTraits: class")

    for helper in ('get_nbserver_extensions', 'diff_members', 'proxy'):
        assert callable(getattr(nbserver, helper)), helper
        print(f"  nbserver.{helper}: callable")

    print("\nExample 1: Compatibility layer exploration completed successfully")

except Exception as e:
    raise AssertionError(
        f"notebook_shim example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Extension point and NotebookApp traits

In [ ]:
# Example 2: Extension point and NotebookApp traits
import importlib

import notebook_shim
from notebook_shim.traits import NotebookAppTraits

print("Example 2: the Jupyter Server extension point and the NotebookApp traits the shim keeps")

points = notebook_shim._jupyter_server_extension_points()
print(f"Extension points: {points}")
assert points == [{'module': 'notebook_shim.nbserver'}], points

# Jupyter Server imports the listed module and calls its link and load hooks
extension_module = importlib.import_module(points[0]['module'])
for hook in ('_link_jupyter_server_extension', '_load_jupyter_server_extension'):
    assert callable(getattr(extension_module, hook)), hook
    print(f"  {hook}: callable")

# Classic notebook options that the shim still accepts as configurable traits
traits = NotebookAppTraits.class_traits(config=True)
print(f"\nConfigurable NotebookApp traits: {sorted(traits)}")
expected = {'enable_mathjax', 'mathjax_url', 'mathjax_config', 'extra_static_paths',
            'extra_template_paths', 'nbserver_extensions', 'quit_button'}
assert expected <= set(traits), expected - set(traits)
assert traits['enable_mathjax'].default_value is True
assert traits['quit_button'].default_value is True
print(f"  enable_mathjax default: {traits['enable_mathjax'].default_value}")
print(f"  mathjax_config default: {traits['mathjax_config'].default_value}")

print("\nExample 2 completed successfully")

## Example 3: Routing NotebookApp options to ServerApp

In [ ]:
# Example 3: Routing NotebookApp options to ServerApp
import logging

from jupyter_server.extension.application import ExtensionApp
from notebook_shim import shim
from traitlets import Bool, TraitError, Unicode
from traitlets.config.loader import Config

print("Example 3: notebook_shim routes classic NotebookApp options to ServerApp")


class DemoExtensionApp(shim.NotebookConfigShimMixin, ExtensionApp):
    """An extension app that used to subclass NotebookApp."""
    name = 'demoextension'
    enable_mathjax = Bool(config=True)   # also a NotebookApp trait
    allow_origin = Unicode(config=True)  # also a ServerApp trait


class ListHandler(logging.Handler):
    """Collects the shim's warnings so the cell can check them."""

    def __init__(self):
        super().__init__()
        self.messages = []

    def emit(self, record):
        self.messages.append(record.getMessage())


app = DemoExtensionApp()
warnings_seen = ListHandler()
app.log.addHandler(warnings_seen)

classic = Config({
    'NotebookApp': {'port': 9999, 'mathjax_config': 'TeX-AMS_HTML'},
    'ServerApp': {'ip': '127.0.0.1'},
})
shimmed = app.shim_config_from_notebook_to_jupyter_server(classic)
print(f"Shimmed config: {shimmed}")

# 'port' moved to ServerApp; 'mathjax_config' is still a NotebookApp option
assert shimmed['ServerApp']['port'] == 9999, shimmed
assert shimmed['ServerApp']['ip'] == '127.0.0.1', shimmed
assert shimmed['NotebookApp']['mathjax_config'] == 'TeX-AMS_HTML', shimmed
assert 'port' not in shimmed['NotebookApp'], shimmed
moved_warning = shim.NBAPP_TO_SVAPP_SHIM_MSG('port')
assert moved_warning in warnings_seen.messages, warnings_seen.messages
print(f"Warning: {moved_warning}")

# An option that exists nowhere is an error, not a silent drop
try:
    app.shim_config_from_notebook_to_jupyter_server(Config({'NotebookApp': {'no_such_option': 1}}))
except TraitError as e:
    print(f"Unknown option rejected: {e}")
else:
    raise AssertionError("an unknown NotebookApp option was accepted")

print("\nExample 3 completed successfully")

## Example 4: Options set on an extension app

In [ ]:
# Example 4: Options set on an extension app
# Reuses DemoExtensionApp, app and warnings_seen from Example 3.
from notebook_shim import shim
from traitlets.config.loader import Config

print("Example 4: options set on the extension app go where the trait lives")

extension_name = DemoExtensionApp.__name__
warnings_seen.messages.clear()
shimmed = app.shim_config_from_notebook_to_jupyter_server(Config({
    extension_name: {
        'enable_mathjax': False,              # extension and NotebookApp: stays on the extension
        'allow_origin': '*',                  # extension and ServerApp: stays on the extension
        'allow_credentials': True,            # ServerApp only: moves to ServerApp
        'mathjax_url': '/static/mathjax.js',  # NotebookApp only: moves to NotebookApp
    }
}))
print(f"Shimmed config: {shimmed}")
assert shimmed[extension_name] == {'enable_mathjax': False, 'allow_origin': '*'}, shimmed
assert shimmed['ServerApp'] == {'allow_credentials': True}, shimmed
assert shimmed['NotebookApp'] == {'mathjax_url': '/static/mathjax.js'}, shimmed

expected_warnings = [
    shim.EXTAPP_AND_NBAPP_SHIM_MSG('enable_mathjax', extension_name),
    shim.EXTAPP_AND_SVAPP_SHIM_MSG('allow_origin', extension_name),
    shim.EXTAPP_TO_SVAPP_SHIM_MSG('allow_credentials', extension_name),
    shim.EXTAPP_TO_NBAPP_SHIM_MSG('mathjax_url', extension_name),
]
for message in expected_warnings:
    assert message in warnings_seen.messages, (message, warnings_seen.messages)
    print(f"  warned: {message[:72]}...")

# update_config() runs the same routing and then applies the values to the app
app.update_config(Config({extension_name: {'enable_mathjax': False}}))
assert app.enable_mathjax is False, app.enable_mathjax
print(f"enable_mathjax after update_config: {app.enable_mathjax}")

print("\nExample 4 completed successfully")

## Example 5: Carrying classic server extensions over

In [ ]:
# Example 5: Carrying classic server extensions over
import json
import os
import tempfile

from notebook_shim import nbserver
from traitlets import Bool, HasTraits, Integer

print("Example 5: reading classic nbserver_extensions config and proxying NotebookApp members")

# get_nbserver_extensions() reads NotebookApp.nbserver_extensions from jupyter_notebook_config files
with tempfile.TemporaryDirectory() as config_dir:
    with open(os.path.join(config_dir, 'jupyter_notebook_config.json'), 'w', encoding='utf-8') as f:
        json.dump({'NotebookApp': {'nbserver_extensions': {'legacy_ext': True, 'disabled_ext': False}}}, f)
    found = nbserver.get_nbserver_extensions([config_dir])
    print(f"Classic server extensions found: {found}")
    assert found == {'legacy_ext': True, 'disabled_ext': False}, found

    empty_dir = os.path.join(config_dir, 'empty')
    os.mkdir(empty_dir)
    assert nbserver.get_nbserver_extensions([empty_dir]) == {}


# proxy() exposes members of one object (the old NotebookApp) on another (the ServerApp)
class Server(HasTraits):
    port = Integer(8888)


class LegacyNotebookApp(HasTraits):
    port = Integer(1)
    quit_button = Bool(True)

    def greet(self, name):
        return f"hello {name}"


server, legacy = Server(), LegacyNotebookApp()
missing = nbserver.diff_members(server, legacy)
print(f"Members only the legacy app has: {sorted(missing)}")
assert missing == {'greet', 'quit_button'}, missing

for name in sorted(missing):
    nbserver.proxy(server, legacy, name)
assert server.greet('shim') == 'hello shim'
assert server.quit_button is True
print(f"Proxied: server.greet('shim') -> {server.greet('shim')!r}, server.quit_button -> {server.quit_button}")

# A member the target already has is never overwritten
try:
    nbserver.proxy(server, legacy, 'port')
except nbserver.ClassProxyError as e:
    print(f"Refused to overwrite: {e}")
else:
    raise AssertionError("proxy() overwrote an existing member")
assert server.port == 8888

print("\nExample 5 completed successfully")